In [2]:
import os
import sys
#print(sys.executable)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import root_scalar
from scipy.integrate import quad
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.integrate import trapezoid
import glob

In [3]:
ref_87 = 384230484468500
P_3_87 = 193740700
P_2_87 = -72911200
P_1_87 = -229851800
P_0_87 = -302073800
S_2_87 = 2563005979.089109
S_1_87 = 4271676631.815181

ref_85 = 384230406373000
P_4_85 = 100357000
P_3_85 = -20503000
P_2_85 = -83955000
P_1_85 = -113307000
S_3_85 = 1264888516.3
S_2_85 = 1770843922.8

# -----------------------------
# Hyperfine constants (MHz)
# -----------------------------
# Rb-87
P_87 = {0: P_0_87, 1: P_1_87, 2: P_2_87, 3: P_3_87}
S_87 = {1: S_1_87, 2: S_2_87}

# Rb-85
P_85 = {1: P_1_85, 2: P_2_85, 3: P_3_85, 4: P_4_85}
S_85 = {2: S_2_85, 3: S_3_85}

# -----------------------------
# 1) Allowed transitions
# -----------------------------
transitions_87 = {}
transitions_85 = {}

# Rb-87: F=1,2 → F'=0,1,2,3
for F in [1, 2]:
    for Fp in [0, 1, 2, 3]:
        if abs(F - Fp) <= 1:
            freq = ref_87 + P_87[Fp] - S_87[F]
            transitions_87[(F, Fp)] = freq

# Rb-85: F=2,3 → F'=1,2,3,4
for F in [2, 3]:
    for Fp in [1, 2, 3, 4]:
        if abs(F - Fp) <= 1:
            freq = ref_85 + P_85[Fp] - S_85[F]
            transitions_85[(F, Fp)] = freq

# -----------------------------
# 2) Crossover transitions
# -----------------------------
cross_87 = {}
cross_85 = {}

# 87 crossovers
keys = list(transitions_87.keys())
for i in range(len(keys)):
    for j in range(i+1, len(keys)):
        (F1, Fp1) = keys[i]
        (F2, Fp2) = keys[j]
        if F1 == F2:
            f1 = transitions_87[(F1, Fp1)]
            f2 = transitions_87[(F2, Fp2)]
            cross_87[((F1, Fp1), (F2, Fp2))] = 0.5*(f1 + f2)

# 85 crossovers
keys = list(transitions_85.keys())
for i in range(len(keys)):
    for j in range(i+1, len(keys)):
        (F1, Fp1) = keys[i]
        (F2, Fp2) = keys[j]
        if F1 == F2:
            f1 = transitions_85[(F1, Fp1)]
            f2 = transitions_85[(F2, Fp2)]
            cross_85[((F1, Fp1), (F2, Fp2))] = 0.5*(f1 + f2)

# -----------------------------
# 3) Print results
# -----------------------------
print("=== Rb-87 Allowed transitions (THz) ===")
for k, v in transitions_87.items():
    print(f"F={k[0]} → F'={k[1]} : {v/1e12} THz")

print("\n=== Rb-87 Crossovers (THz) ===")
for k, v in cross_87.items():
    (F1, Fp1), (F2, Fp2) = k
    print(f"(F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2}) : {v/1e12} THz")

print("\n=== Rb-85 Allowed transitions (THz) ===")
for k, v in transitions_85.items():
    print(f"F={k[0]} → F'={k[1]} : {v/1e12} THz")

print("\n=== Rb-85 Crossovers (THz) ===")
for k, v in cross_85.items():
    (F1, Fp1), (F2, Fp2) = k
    print(f"(F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2}) : {v/1e12} THz")


=== Rb-87 Allowed transitions (THz) ===
F=1 → F'=0 : 384.2259107180682 THz
F=1 → F'=1 : 384.2259829400682 THz
F=1 → F'=2 : 384.2261398806682 THz
F=2 → F'=1 : 384.2276916107209 THz
F=2 → F'=2 : 384.22784855132096 THz
F=2 → F'=3 : 384.2281152032209 THz

=== Rb-87 Crossovers (THz) ===
(F=1→F'=0) × (F=1→F'=1) : 384.22594682906816 THz
(F=1→F'=0) × (F=1→F'=2) : 384.2260252993682 THz
(F=1→F'=1) × (F=1→F'=2) : 384.2260614103682 THz
(F=2→F'=1) × (F=2→F'=2) : 384.22777008102094 THz
(F=2→F'=1) × (F=2→F'=3) : 384.22790340697094 THz
(F=2→F'=2) × (F=2→F'=3) : 384.22798187727096 THz

=== Rb-85 Allowed transitions (THz) ===
F=2 → F'=1 : 384.2285222220772 THz
F=2 → F'=2 : 384.2285515740772 THz
F=2 → F'=3 : 384.2286150260772 THz
F=3 → F'=2 : 384.22905752948367 THz
F=3 → F'=3 : 384.2291209814837 THz
F=3 → F'=4 : 384.2292418414837 THz

=== Rb-85 Crossovers (THz) ===
(F=2→F'=1) × (F=2→F'=2) : 384.2285368980772 THz
(F=2→F'=1) × (F=2→F'=3) : 384.2285686240772 THz
(F=2→F'=2) × (F=2→F'=3) : 384.22858330007716 

In [4]:
all_lines = []

# -----------------------------
# 1) Allowed transitions (87)
# -----------------------------
for (F, Fp), v in transitions_87.items():
    freq = v
    label = f"Rb87: F={F} → F'={Fp}"
    all_lines.append((freq, label))

# -----------------------------
# 2) Crossover transitions (87)
# -----------------------------
for k, v in cross_87.items():
    (F1, Fp1), (F2, Fp2) = k
    freq = v
    label = f"Rb87: (F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2})"
    all_lines.append((freq, label))

# -----------------------------
# 3) Allowed transitions (85)
# -----------------------------
for (F, Fp), v in transitions_85.items():
    freq = v
    label = f"Rb85: F={F} → F'={Fp}"
    all_lines.append((freq, label))

# -----------------------------
# 4) Crossover transitions (85)
# -----------------------------
for k, v in cross_85.items():
    (F1, Fp1), (F2, Fp2) = k
    freq = v
    label = f"Rb85: (F={F1}→F'={Fp1}) × (F={F2}→F'={Fp2})"
    all_lines.append((freq, label))

# -----------------------------
# 5) Sort by frequency
# -----------------------------
all_lines_sorted = sorted(all_lines, key=lambda x: x[0])

# -----------------------------
# 6) Print
# -----------------------------
print("=== All transitions (Rb-87 + Rb-85, real + crossover) sorted by frequency (THz) ===")
for freq, label in all_lines_sorted:
    print(f"{freq/1e12} THz  :  {label}")

=== All transitions (Rb-87 + Rb-85, real + crossover) sorted by frequency (THz) ===
384.2259107180682 THz  :  Rb87: F=1 → F'=0
384.22594682906816 THz  :  Rb87: (F=1→F'=0) × (F=1→F'=1)
384.2259829400682 THz  :  Rb87: F=1 → F'=1
384.2260252993682 THz  :  Rb87: (F=1→F'=0) × (F=1→F'=2)
384.2260614103682 THz  :  Rb87: (F=1→F'=1) × (F=1→F'=2)
384.2261398806682 THz  :  Rb87: F=1 → F'=2
384.2276916107209 THz  :  Rb87: F=2 → F'=1
384.22777008102094 THz  :  Rb87: (F=2→F'=1) × (F=2→F'=2)
384.22784855132096 THz  :  Rb87: F=2 → F'=2
384.22790340697094 THz  :  Rb87: (F=2→F'=1) × (F=2→F'=3)
384.22798187727096 THz  :  Rb87: (F=2→F'=2) × (F=2→F'=3)
384.2281152032209 THz  :  Rb87: F=2 → F'=3
384.2285222220772 THz  :  Rb85: F=2 → F'=1
384.2285368980772 THz  :  Rb85: (F=2→F'=1) × (F=2→F'=2)
384.2285515740772 THz  :  Rb85: F=2 → F'=2
384.2285686240772 THz  :  Rb85: (F=2→F'=1) × (F=2→F'=3)
384.22858330007716 THz  :  Rb85: (F=2→F'=2) × (F=2→F'=3)
384.2286150260772 THz  :  Rb85: F=2 → F'=3
384.22905752948367 

In [5]:
rb87 = []
rb85 = []

for freq, label in all_lines_sorted:
    if label.startswith("Rb87"):
        rb87.append((freq, label))
    elif label.startswith("Rb85"):
        rb85.append((freq, label))
print(rb85)

[(384228522222077.2, "Rb85: F=2 → F'=1"), (384228536898077.2, "Rb85: (F=2→F'=1) × (F=2→F'=2)"), (384228551574077.2, "Rb85: F=2 → F'=2"), (384228568624077.2, "Rb85: (F=2→F'=1) × (F=2→F'=3)"), (384228583300077.2, "Rb85: (F=2→F'=2) × (F=2→F'=3)"), (384228615026077.2, "Rb85: F=2 → F'=3"), (384229057529483.7, "Rb85: F=3 → F'=2"), (384229089255483.7, "Rb85: (F=3→F'=2) × (F=3→F'=3)"), (384229120981483.7, "Rb85: F=3 → F'=3"), (384229149685483.7, "Rb85: (F=3→F'=2) × (F=3→F'=4)"), (384229181411483.7, "Rb85: (F=3→F'=3) × (F=3→F'=4)"), (384229241841483.7, "Rb85: F=3 → F'=4")]


In [6]:

def compute_normalized_intervals(group):
    freqs = np.array([f for f, lab in group])
    labels = [lab for f, lab in group]

    intervals = []  # (norm_df, df, label_i, label_j)

    # 모든 조합 (i < j)
    for i in range(len(freqs)):
        for j in range(i+1, len(freqs)):
            df = freqs[j] - freqs[i]
            intervals.append((df, labels[i], labels[j]))

    # 간격만 추출해서 정규화
    df_values = np.array([x[0] for x in intervals])
    max_df = df_values.max()

    normalized = []
    for df, lab1, lab2 in intervals:
        norm_df = df / max_df
        normalized.append((norm_df, df, lab1, lab2))

    # 정규화된 간격 순으로 정렬
    normalized.sort(key=lambda x: x[0])
    return normalized

norm_intervals_87 = compute_normalized_intervals(rb87)
norm_intervals_85 = compute_normalized_intervals(rb85)
print("=== Rb-87 normalized intervals ===")
for norm_df, df, lab1, lab2 in norm_intervals_87:
    print(f"norm={norm_df:.6f}   df={df/1e12:.9f} THz   :  {lab1} ↔ {lab2}")

print("\n=== Rb-85 normalized intervals ===")
for norm_df, df, lab1, lab2 in norm_intervals_85:
    print(f"norm={norm_df:.6f}   df={df/1e12:.9f} THz   :  {lab1} ↔ {lab2}")

=== Rb-87 normalized intervals ===
norm=0.016381   df=0.000036111 THz   :  Rb87: F=1 → F'=0 ↔ Rb87: (F=1→F'=0) × (F=1→F'=1)
norm=0.016381   df=0.000036111 THz   :  Rb87: (F=1→F'=0) × (F=1→F'=1) ↔ Rb87: F=1 → F'=1
norm=0.016381   df=0.000036111 THz   :  Rb87: (F=1→F'=0) × (F=1→F'=2) ↔ Rb87: (F=1→F'=1) × (F=1→F'=2)
norm=0.019215   df=0.000042359 THz   :  Rb87: F=1 → F'=1 ↔ Rb87: (F=1→F'=0) × (F=1→F'=2)
norm=0.024884   df=0.000054856 THz   :  Rb87: F=2 → F'=2 ↔ Rb87: (F=2→F'=1) × (F=2→F'=3)
norm=0.032761   df=0.000072222 THz   :  Rb87: F=1 → F'=0 ↔ Rb87: F=1 → F'=1
norm=0.035596   df=0.000078470 THz   :  Rb87: (F=1→F'=0) × (F=1→F'=1) ↔ Rb87: (F=1→F'=0) × (F=1→F'=2)
norm=0.035596   df=0.000078470 THz   :  Rb87: F=1 → F'=1 ↔ Rb87: (F=1→F'=1) × (F=1→F'=2)
norm=0.035596   df=0.000078470 THz   :  Rb87: (F=1→F'=1) × (F=1→F'=2) ↔ Rb87: F=1 → F'=2
norm=0.035596   df=0.000078470 THz   :  Rb87: F=2 → F'=1 ↔ Rb87: (F=2→F'=1) × (F=2→F'=2)
norm=0.035596   df=0.000078470 THz   :  Rb87: (F=2→F'=1) × (F=

In [7]:
data4_T = [0.0114427, 0.0120711, 0.012397, 0.0126802, 0.0130316]
data4_1 = [0.2874390050049227, 0.33302702376055976, 0.31881418486575, 0.2970975120096744, 0.3008597511899701]
data4_2 = [2.06891, 2.07812, 2.07953, 2.07188, 2.05031]
data6_T = [0.011453, 0.0120791, 0.0124029, 0.0130008]
data6_1 = [-0.72052778409326, -1.8063276656924325, -1.3524347551805662, -0.5659081251481841 ]
data6_2 = [2.0748070597035704, 2.0749573162771995, 2.076009878367027, 2.075763212476051]


In [8]:
from itertools import combinations

# -----------------------------
# 1) Experimental ratio
# -----------------------------
exp = np.array(data6_T)
exp = np.sort(exp)
exp_d = np.diff(exp)
exp_ratio = exp_d / exp_d[0]   # 3개 비율
print("Experimental ratio:", exp_ratio)

# -----------------------------
# 2) 함수: 4개 전이 조합에서 간격 비율 비교
# -----------------------------
def find_matching_groups(group, exp_ratio, tol=0.05):
    matches = []
    freqs = np.array([f for f, lab in group])
    labels = [lab for f, lab in group]

    for idx in combinations(range(len(freqs)), 4):
        block = freqs[list(idx)]
        block_labels = [labels[i] for i in idx]

        block = np.sort(block)
        d = np.diff(block)

        if d[0] == 0:
            continue

        ratio = d / d[0]
        rel_err = np.abs(ratio - exp_ratio) / exp_ratio

        if np.all(rel_err < tol):
            matches.append((block, block_labels, ratio))

    return matches


matches_87 = find_matching_groups(rb87, exp_ratio)
matches_85 = find_matching_groups(rb85, exp_ratio)
print("=== Matches in Rb-87 ===")
for block, labels, ratio in matches_87:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

print("\n=== Matches in Rb-85 ===")
for block, labels, ratio in matches_85:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

Experimental ratio: [1.         0.51716978 0.95495927]
=== Matches in Rb-87 ===

=== Matches in Rb-85 ===

Matched ratio: [1.         0.52500414 1.        ]
  384.229089 THz : Rb85: (F=3→F'=2) × (F=3→F'=3)
  384.229150 THz : Rb85: (F=3→F'=2) × (F=3→F'=4)
  384.229181 THz : Rb85: (F=3→F'=3) × (F=3→F'=4)
  384.229242 THz : Rb85: F=3 → F'=4


In [12]:

from itertools import combinations

# -----------------------------
# 1) Experimental ratio (generalized)
# -----------------------------
data4_T = [0.0114427, 0.0120711, 0.012397, 0.0130316]
exp = np.array(data4_T)


exp = np.sort(exp)
exp_d = np.diff(exp)              # N-1개 간격
exp_ratio = exp_d / exp_d[0]      # 비율
N = len(exp)                      # 실험 포인트 개수

print("Experimental ratio:", exp_ratio)

# -----------------------------
# 2) Generalized matching function
# -----------------------------
def find_matching_groups(group, exp_ratio, N, tol=0.05):
    matches = []
    freqs = np.array([f for f, lab in group])
    labels = [lab for f, lab in group]

    # 이론 전이에서 N개씩 조합
    for idx in combinations(range(len(freqs)), N):
        block = freqs[list(idx)]
        block_labels = [labels[i] for i in idx]

        block = np.sort(block)
        d = np.diff(block)        # N-1개 간격

        if d[0] == 0:
            continue

        ratio = d / d[0]
        rel_err = np.abs(ratio - exp_ratio) / exp_ratio

        if np.all(rel_err < tol):
            matches.append((block, block_labels, ratio))

    return matches



matches_87 = find_matching_groups(rb87, exp_ratio, N)
matches_85 = find_matching_groups(rb85, exp_ratio, N)
print("=== Matches in Rb-87 ===")
for block, labels, ratio in matches_87:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

print("\n=== Matches in Rb-85 ===")
for block, labels, ratio in matches_85:
    print("\nMatched ratio:", ratio)
    for f, lab in zip(block, labels):
        print(f"  {f/1e12:.6f} THz : {lab}")

Experimental ratio: [1.         0.51861871 1.00986633]
=== Matches in Rb-87 ===

=== Matches in Rb-85 ===

Matched ratio: [1.         0.52500414 1.        ]
  384.229089 THz : Rb85: (F=3→F'=2) × (F=3→F'=3)
  384.229150 THz : Rb85: (F=3→F'=2) × (F=3→F'=4)
  384.229181 THz : Rb85: (F=3→F'=3) × (F=3→F'=4)
  384.229242 THz : Rb85: F=3 → F'=4
